# Distillation Workflow: Grok Collection

Collect teacher-generated images using Grok (xAI) from prompts in `distill/prompts.csv`.


In [11]:
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))


## 1. Environment Setup

In [12]:
import os
from distill.env import load_dotenv_if_present

load_dotenv_if_present()

has_key = bool(os.environ.get("OPENROUTER_API_KEY"))
print(f"XAI/GROK: {'✅' if has_key else '❌'}")
if not has_key:
    print("Add OPENROUTER_API_KEY to .env")


OPENROUTER_API_KEY: ✅


## 2. Load Prompts

In [13]:
from distill.prompts import default_prompts

all_prompts = default_prompts()
print(f"Loaded {len(all_prompts)} prompts")


Loaded 89 prompts


## 3. Configure Collection

In [15]:
from distill.config import DistillConfig, apply_model_preset

RUN_ID = "grok_v1"
TEACHER = "grok"
SAMPLES_PER_PROMPT = 1
MAX_PROMPTS = 50
MAX_CONCURRENT = 4
PROCEED = True

cfg = DistillConfig(
    run_id=RUN_ID,
    dataset_name="grok",
    teacher=TEACHER,
    model_preset="sd15",
    samples_per_prompt=SAMPLES_PER_PROMPT,
    max_samples=MAX_PROMPTS,
    max_concurrent=MAX_CONCURRENT,
    partial_supervision=False,
)
cfg = apply_model_preset(cfg)

prompts_to_use = all_prompts[:MAX_PROMPTS]
total_calls = len(prompts_to_use) * SAMPLES_PER_PROMPT

print(f"Teacher: {TEACHER}")
print(f"Prompts: {len(prompts_to_use)}")
print(f"Samples/prompt: {SAMPLES_PER_PROMPT}")
print(f"Total calls: {total_calls}")
print(f"PROCEED: {PROCEED}")


Teacher: openrouter
Prompts: 50
Samples/prompt: 1
Total calls: 50
PROCEED: True


## 4. Run Collection

In [16]:
import time
import asyncio
import nest_asyncio; nest_asyncio.apply()

if PROCEED:
    from distill.collect import collect
    
    start = time.time()
    manifest = await collect(cfg)
    print(f"Done in {time.time()-start:.1f}s")
    print(f"Manifest: {manifest}")
else:
    print("Set PROCEED = True to run")


ClientConnectorDNSError: Cannot connect to host api.openrouter.ai:443 ssl:default [Domain name not found]

## 5. Inspect Results

In [17]:
import json
from IPython.display import display, Image
from pathlib import Path

root = Path(cfg.output_dir) / cfg.dataset_name / cfg.run_id
manifest_path = root / "manifest.jsonl"

if manifest_path.exists():
    with open(manifest_path) as f:
        records = [json.loads(l) for l in f]
    print(f"Records: {len(records)}")
    for rec in records[:6]:
        print(rec.get('prompt', ''))
        if Path(rec.get('final_path','')).exists():
            display(Image(rec['final_path']))
else:
    print("No manifest found")


No manifest found


## 6. Cleanup

In [ ]:
print("Done. Run `python -m client.manage clearmem <provider> --free-model` if using remote GPU.")
